In [ ]:
import random
import os

# ================================================================== #
# Configuration                                                        #
# ================================================================== #

# ---- Byte premium factors (information capacity per token) ----
BYTE_PREMIUM_ENGLISH = 1.000000
BYTE_PREMIUM_DUTCH   = 1.051606
BYTE_PREMIUM_CHINESE = 0.935966

# ---- Token budget ----
TOKEN_TARGET = 30_000_000

# ---- Per-language ratio (must sum to 1.0) ----
RATIO_ENG = 1/3
RATIO_NLD = 1/3
RATIO_ZHO = 1/3

# ---- Per-language adjusted budgets ----
# Divide by byte premium so each language contributes equal *information*.
BUDGET_ENG = round((TOKEN_TARGET * RATIO_ENG) / BYTE_PREMIUM_ENGLISH)
BUDGET_NLD = round((TOKEN_TARGET * RATIO_NLD) / BYTE_PREMIUM_DUTCH)
BUDGET_ZHO = round((TOKEN_TARGET * RATIO_ZHO) / BYTE_PREMIUM_CHINESE)

# ---- Source files ----
INPUT_FILES = {
    "eng": [
        "data/en-nl/OpenSubtitles.en-nl.en",
        "data/en-zh/OpenSubtitles.en-zh.en",
    ],
    "nld": ["data/en-nl/OpenSubtitles.en-nl.nl"],
    "zho": ["data/en-zh/OpenSubtitles.en-zh.zh"],
}

LANG_BUDGETS = {
    "eng": BUDGET_ENG,
    "nld": BUDGET_NLD,
    "zho": BUDGET_ZHO,
}

# ---- Output paths ----
OUTPUT_TRAIN      = "./data/opensubs_30m_train.txt"
OUTPUT_VALIDATION = "./data/opensubs_30m_validation.txt"

# ---- Misc ----
VALIDATION_RATIO = 0.1
SEED             = 42


# ================================================================== #
# Helpers                                                              #
# ================================================================== #

def read_lines_to_budget(filepaths: list[str], token_budget: int) -> tuple[list[str], int]:
    """
    Read sentences from one or more files until *token_budget* whitespace
    tokens have been collected.  Returns (lines, actual_token_count).
    """
    lines: list[str] = []
    total = 0
    for filepath in filepaths:
        if total >= token_budget:
            break
        with open(filepath, encoding="utf-8") as f:
            for raw in f:
                words = raw.split()
                if not words:
                    continue
                n = len(words)
                if total + n <= token_budget:
                    lines.append(raw.strip())
                    total += n
                else:
                    break
    return lines, total


# ================================================================== #
# Main                                                                 #
# ================================================================== #

def create_opensubs_dataset():
    print("Per-language budgets (byte-premium adjusted):")
    for lang, budget in LANG_BUDGETS.items():
        print(f"  {lang}: {budget:,} tokens")
    print(f"  Total: {sum(LANG_BUDGETS.values()):,} tokens\n")

    # --- Read each language to its budget ---
    all_lines: list[str] = []
    for lang, filepaths in INPUT_FILES.items():
        budget = LANG_BUDGETS[lang]
        lines, actual = read_lines_to_budget(filepaths, budget)
        print(f"{lang}: {actual:,} tokens from {len(lines):,} lines")
        all_lines.extend(lines)

    # --- Shuffle and split ---
    print(f"\nTotal: {len(all_lines):,} lines")
    random.seed(SEED)
    random.shuffle(all_lines)

    split_idx   = int((1.0 - VALIDATION_RATIO) * len(all_lines))
    train_lines = all_lines[:split_idx]
    val_lines   = all_lines[split_idx:]

    # --- Write ---
    os.makedirs(os.path.dirname(OUTPUT_TRAIN) or ".", exist_ok=True)

    with open(OUTPUT_TRAIN, "w", encoding="utf-8") as f:
        for line in train_lines:
            f.write(line + "\n")

    with open(OUTPUT_VALIDATION, "w", encoding="utf-8") as f:
        for line in val_lines:
            f.write(line + "\n")

    print(f"\nDone!")
    print(f"  Train:      {OUTPUT_TRAIN}  ({len(train_lines):,} lines)")
    print(f"  Validation: {OUTPUT_VALIDATION}  ({len(val_lines):,} lines)")


In [2]:
create_opensubs_dataset()

Per-language budgets (byte-premium adjusted):
  eng: 10,000,000 tokens
  nld: 9,509,265 tokens
  zho: 10,684,149 tokens
  Total: 30,193,414 tokens

eng: 9,999,996 tokens from 1,448,471 lines
nld: 9,509,259 tokens from 1,563,851 lines
zho: 10,684,149 tokens from 6,819,705 lines

Total: 9,832,027 lines

Done!
  Train:      ./data/opensubs_30m_train.txt  (8,848,824 lines)
  Validation: ./data/opensubs_30m_validation.txt  (983,203 lines)
